Data preparation update: the kids group is excluded, splits use stratified 80/10/10 sampling, and CSVs omit the row index. Historical discussion below describes earlier experiments; rerun cells for current metrics. Outputs were cleared to avoid presenting old-split results as current.


In [ ]:
from pathlib import Path
import pandas as pd

# Support running from the project root or the project_checkpoint folder.
train_path = Path("data/train.csv")
if not train_path.exists():
    train_path = Path("../data/train.csv")

# Preprocessing saves CSVs without a row index.
train_df = pd.read_csv(train_path)
train_df.head()
len(train_df)

In [ ]:
# Select the audio features and the genre label to predict.
features = [
    "danceability", "energy", "loudness", "speechiness",
    "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "duration_ms",
]

X = train_df[features].copy()
y = train_df["genre_group"].copy()

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale each feature before passing it to the classifier.
model = Pipeline([
    ("scaler", StandardScaler()), # scales each feature by subtracting by training mean and dividing by training std
    ("logistic_regression", LogisticRegression(solver="lbfgs", max_iter=2000)), # creates classifier, training stops at 2000 iterations
])

model


In [ ]:
# Fit the scaler and logistic regression using the training data.
model.fit(X, y)


In [ ]:
import joblib # helps us use our trained model later on, without having to retrain 

# Check whether training reached the iteration limit.
classifier = model.named_steps["logistic_regression"] # fetches the logistic regression model created above 
print("Optimization iterations:", classifier.n_iter_) # this prints the number of iterations needed 
if classifier.n_iter_.max() >= classifier.max_iter:
    print("Iteration limit reached; review any convergence warning from training.")
else:
    print("Training stopped before the iteration limit.")

# This is a training sanity check, not an estimate of unseen-data performance.
print(f"Training accuracy: {model.score(X, y):.3f}")

# Save both the fitted scaler and classifier together.
model_path = train_path.resolve().parent.parent / "project_checkpoint" / "logistic-regression.joblib"
joblib.dump(model, model_path) # saves the entire fitted model 
print(f"Saved pipeline to: {model_path}")


A training accuracy of 40% means the model correctly predicts the genre group for about 40 out of every 100 training tracks. These are tracks it already learned from, so this does not measure performance on unseen tracks.

Compare this with the majority-class baseline: always predicting the most frequent genre achieves accuracy equal to that genre's proportion. For example, if that proportion is 35%, the baseline accuracy is 35% (illustrative, not yet measured here).

Preprocessing splits the held-out data equally into validation and final test CSVs. Use validation to compare models and keep the final test set untouched until model choices are finalized.


In [ ]:
from sklearn.metrics import accuracy_score

# Preprocessing creates the separate validation and final test CSVs.
validation_df = pd.read_csv(train_path.parent / "validation.csv")
X_validation = validation_df[features].copy()
y_validation = validation_df["genre_group"].copy()

print("Validation rows:", len(validation_df))

# Reuse the training-fitted scaler and classifier without fitting again.
validation_predictions = model.predict(X_validation)
print(f"Validation accuracy: {accuracy_score(y_validation, validation_predictions):.2%}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Inspect precision, recall, and F1 for each genre on validation data.
print("Validation classification report:")
print(classification_report(
    y_validation, validation_predictions,
    labels=model.classes_, zero_division=0,
))

# Rows are actual genres; columns are predicted genres.
validation_confusion_matrix = pd.DataFrame(
    confusion_matrix(y_validation, validation_predictions, labels=model.classes_),
    index=model.classes_,
    columns=model.classes_,
)
validation_confusion_matrix.index.name = "Actual genre"
validation_confusion_matrix.columns.name = "Predicted genre"
print("Validation confusion matrix (counts):")
print(validation_confusion_matrix.to_string())


For now, we can see the training and validation accurary both are at around 40% 
This means that our model isn't overfitting, but it still doesn't beat a 50% accuracy of random guessing. A good next step could be trying to add more features 

In [ ]:
# Keep the original features and add four attributes for the next experiment.
# Key and time_signature will need categorical encoding in a later step.
expanded_features = features + [
    "mode",
    "key",
    "time_signature",
    "explicit",
]

print("Expanded feature list:", expanded_features)
print("Number of features:", len(expanded_features))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

X_expanded = train_df[expanded_features].copy()

# Learn scaling and categories from training data only.
expanded_preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), ["key", "time_signature"]),
    ("binary", "passthrough", ["mode", "explicit"]),
]) # new scaling for the added features 
# since the type of scaling is different for different features, we use ColumnTransformer

# Keep the original model available for comparison.
expanded_model = Pipeline([
    ("preprocessor", expanded_preprocessor),
    ("logistic_regression", LogisticRegression(solver="lbfgs", max_iter=2000)),
])

expanded_model.fit(X_expanded, y)
print("Expanded-feature model training complete.")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Both models predict the same validation tracks using their fitted preprocessing.
X_validation_expanded = validation_df[expanded_features].copy()
expanded_validation_predictions = expanded_model.predict(X_validation_expanded)
original_validation_predictions = model.predict(X_validation)

original_accuracy = accuracy_score(y_validation, original_validation_predictions)
expanded_accuracy = accuracy_score(y_validation, expanded_validation_predictions)

print(f"Original model validation accuracy: {original_accuracy:.2%}")
print(f"Expanded model validation accuracy: {expanded_accuracy:.2%}")
print(f"Accuracy change: {(expanded_accuracy - original_accuracy) * 100:+.2f} percentage points")
print(f"Original model validation macro F1: {f1_score(y_validation, original_validation_predictions, average='macro', zero_division=0):.3f}")
print(f"Expanded model validation macro F1: {f1_score(y_validation, expanded_validation_predictions, average='macro', zero_division=0):.3f}")
print(f"Expanded model training accuracy: {expanded_model.score(X_expanded, y):.2%}")


Since adding features doesn't bring about a huge difference in the train and validation accuracy, overfitting does not seem to be an issue. We can hold off on feature reduction. 

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score

# Change only regularization; refit preprocessing on training data for each C.
C_values = [0.01, 0.1, 1, 10, 100]
regularization_results = []
regularized_models = {}
X_validation_expanded = validation_df[expanded_features].copy()

for C in C_values:
    candidate = clone(expanded_model)
    candidate.set_params(logistic_regression__C=C)
    candidate.fit(X_expanded, y)
    predictions = candidate.predict(X_validation_expanded)
    classifier = candidate.named_steps["logistic_regression"]
    regularization_results.append({
        "C": C,
        "training_accuracy": candidate.score(X_expanded, y),
        "validation_accuracy": accuracy_score(y_validation, predictions),
        "validation_macro_f1": f1_score(
            y_validation, predictions, average="macro", zero_division=0,
        ),
        "iterations": int(classifier.n_iter_.max()),
        "iteration_limit_reached": bool(classifier.n_iter_.max() >= classifier.max_iter),
    })
    regularized_models[C] = candidate

regularization_results = pd.DataFrame(regularization_results)
print(regularization_results.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

# Select by validation accuracy, then macro F1; retain the original expanded model.
best_row = regularization_results.sort_values(
    ["validation_accuracy", "validation_macro_f1"], ascending=False,
).iloc[0]
best_C = float(best_row["C"])
best_regularized_model = regularized_models[best_C]
print(f"Selected C: {best_C:g}")
print(f"Selected validation accuracy: {best_row['validation_accuracy']:.2%}")
print(f"Selected validation macro F1: {best_row['validation_macro_f1']:.3f}")
# Validation guided this choice; the final test set remains reserved.


The results didn't change much even though changes were made in the regularization parameter. 